# torch.distributed.broadcast

torch.distributed.broadcast 是 PyTorch 分布式通信包（torch.distributed）中最基础的集合通信（Collective Communication）原语之一，用于将某个进程（源进程）上的张量数据广播到同一进程组内的所有其他进程。它是构建分布式数据并行（DDP）、模型参数同步、随机种子一致性等功能的基石。

## 基本概念
- 广播（Broadcast） 的通信模式是：进程组中指定一个源进程（src），该进程将其张量数据发送给组内所有其他进程。通信完成后，组内每个进程持有的张量在位级别（bitwise）上完全一致 


In [ ]:
Rank 0 (src)  ──tensor──►  Rank 1
              ──tensor──►  Rank 2
              ──tensor──►  Rank 3

##  API 签名与参数详解

In [ ]:
torch.distributed.broadcast(
    tensor,           # Tensor
    src=None,         # int
    group=None,       # ProcessGroup, optional
    async_op=False,   # bool, optional
    group_src=None    # int
)

| 参数              | 说明                                                                                      |
| --------------- | --------------------------------------------------------------------------------------- |
| **`tensor`**    | 在 **src** 进程上为**待发送的数据**；在其他进程上为**接收缓冲区**（用于保存接收到的数据）。所有参与进程的张量必须具有**相同数量的元素** 。        |
| **`src`**       | 源进程在**全局进程组**中的 rank（不受 `group` 参数影响）。                                                  |
| **`group`**     | 指定操作的进程组（`ProcessGroup`）。若为 `None`，则使用默认进程组 。                                           |
| **`async_op`**  | 是否为异步操作。若为 `True`，函数立即返回一个异步工作句柄（`Work` 对象），需手动调用 `.wait()` 完成同步；若为 `False`，则阻塞直到广播完成 。 |
| **`group_src`** | 源进程在**指定 `group`** 内的 rank。`group_src` 和 `src` **必须且只能指定其中一个** 。                        |

- 返回值：若 async_op=True，返回异步工作句柄；否则返回 None

## 工作原理与底层实现
###  通过 c10d 后端执行

torch.distributed.broadcast 并非直接操作网络，而是调用 PyTorch 内部的 c10d（Communication Library） 层。c10d 根据初始化时选择的 backend 将操作分发到具体的通信库：

| 后端       | 适用场景              | 底层实现                             |
| -------- | ----------------- | -------------------------------- |
| **NCCL** | GPU 张量、单机/多机多卡    | 调用 NVIDIA NCCL 的 `ncclBroadcast` |
| **Gloo** | CPU 张量、开发调试       | Gloo 的广播算法                       |
| **MPI**  | 超算集群、与现有 MPI 程序集成 | MPI 的 `MPI_Bcast`                |

 <font color='red'>当使用 NCCL 后端时，tensor 必须位于 GPU 上（通常为当前 rank 对应的 CUDA 设备），NCCL 会利用 NVLink、NVSwitch 或 InfiniBand 等高速互连完成传输，实现高带宽、低延迟的广播 </font>
 
### 同步机制
- 同步模式（async_op=False）：所有进程在广播完成前阻塞，确保后续代码执行时数据已全局一致。
- 异步模式（async_op=True）：发起广播后立即返回，可与计算重叠（overlap），最后通过句柄 .wait() 同步。这在流水线并行或大规模训练中尤为重要。

## 使用示例
### 基础用法：同步初始模型参数

确保所有进程从 rank 0 加载的相同权重开始训练：

In [ ]:
import torch
import torch.distributed as dist
import torch.multiprocessing as mp

def run(rank, world_size):
    dist.init_process_group("nccl", rank=rank, world_size=world_size)
    torch.cuda.set_device(rank)
    
    # 仅在 rank 0 创建/加载模型
    if rank == 0:
        '''
        使用broadcast广播的数据
        '''
        state_dict = torch.load("model.pth")
    else:
        '''
        使用broadcast接收数据并保存到state_dict中
        '''
        state_dict = {k: torch.empty_like(v) for k, v in ref_state_dict.items()}
    
    # 实际使用时通常直接 broadcast 参数张量
    tensor = torch.tensor([1.0, 2.0, 3.0], device=f"cuda:{rank}")
    if rank != 0:
        tensor.zero_()  # 非 src 进程先准备接收缓冲区
    
    # 从 rank 0 广播到所有进程
    dist.broadcast(tensor, src=0)
    
    print(f"Rank {rank}: {tensor}")  # 所有 rank 输出 [1., 2., 3.]
    dist.destroy_process_group()

if __name__ == "__main__":
    world_size = 4
    mp.spawn(run, args=(world_size,), nprocs=world_size, join=True)

### 异步广播

In [ ]:
handle = dist.broadcast(tensor, src=0, async_op=True)
# ... 在此处执行不依赖该 tensor 的计算 ...
handle.wait()  # 确保广播完成后再使用 tensor

| 注意点                     | 说明                                                                     |
| ----------------------- | ---------------------------------------------------------------------- |
| **张量形状一致**              | 所有参与进程的 `tensor` 必须具有**相同的元素数量**（`numel`），但形状（shape）可以不同（底层按扁平化数据广播）。  |
| **原地操作**                | 广播完成后，非 src 进程的 `tensor` 内容被**覆盖**为 src 进程的数据。                         |
| **src 与 group\_src 互斥** | 若指定了自定义 `group`，可通过 `group_src` 指定组内源 rank；否则使用全局 `src`。二者不可同时指定 。     |
| **NCCL 后端设备匹配**         | 使用 NCCL 时，确保 `tensor` 位于正确的 GPU 上，通常需配合 `torch.cuda.set_device(rank)`。 |
| **进程组一致性**              | 组内所有进程必须**同时调用** `broadcast`，否则会导致死锁。                                  |


## 相关变体 API
###  broadcast_object_list
- 用于广播任意可序列化（picklable）的 Python 对象（如字符串、字典、超参配置等），而非仅限于张量

- 该函数始终阻塞，不提供 async_op 选项 。
- 内部使用 pickle 序列化，存在安全风险，且对 GPU 张量效率低下（会触发 GPU→CPU 传输）

In [ ]:
objects = ["config", {"lr": 0.01}, 42] if rank == 0 else [None, None, None]
dist.broadcast_object_list(objects, src=0)
# 所有 rank 的 objects 变为 ["config", {"lr": 0.01}, 42]

### broadcast_multigpu
- 针对单节点多 GPU 场景，允许将一个张量广播到当前进程的多个 GPU 上

In [ ]:
dist.broadcast_multigpu(tensor_list, src=0, src_tensor=0)
# tensor_list 中每个张量位于不同 GPU，src_tensor 指定列表中哪个是源张量

| 场景            | 说明                                                     |
| ------------- | ------------------------------------------------------ |
| **模型权重初始化同步** | 仅 rank 0 加载预训练权重，通过 `broadcast` 分发给所有 rank，确保各进程起点一致 。 |
| **随机种子同步**    | rank 0 生成随机种子后广播，保证所有进程数据增强、Dropout 等行为一致。             |
| **超参数/配置分发**  | 结合 `broadcast_object_list`，将解析后的配置对象分发给所有工作进程。         |
| **参数服务器模式**   | 某些自定义并行策略中，用 broadcast 替代 all-reduce 实现参数更新分发 。        |


## 与 NCCL 的关系

如前文对 NCCL 的介绍，torch.distributed.broadcast 在 GPU 场景下通常通过 NCCL 后端执行。此时：
- PyTorch 的 broadcast API 是高层抽象，处理张量布局、设备管理、进程组逻辑。
- NCCL 的 ncclBroadcast 是底层实现，负责拓扑感知、协议选择（LL/LL128/Simple）、多流调度等。

两者关系可概括为：
- PyTorch 提供易用的 Python API 和与训练框架的集成，NCCL 提供极致的 GPU 通信性能。
- <font color='red'>在绝大多数生产环境的分布式训练中，开发者调用 dist.broadcast(..., async_op=True)，底层即由 NCCL 在 NVLink/InfiniBand 上完成高效传输。</font>